In [6]:
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

# Cấu hình chung cho font và màu sắc chuyên nghiệp
THEME_COLORS = {
    'background': '#f9f9f9',
    'text': '#2c3e50',
    'funnel': ['#1a2a6c', '#b21f1f', '#fdbb2d', '#2ecc71', '#3498db'],
    'lost': '#e74c3c',
    'success': '#2ecc71'
}

# =======================================================================
# 1. BIỂU ĐỒ PHỄU HIỆN ĐẠI (MODERN FUNNEL)
# =======================================================================
stages_labels = ["1. Truy cập", "2. Xem sản phẩm", "3. Thêm vào giỏ", "4. Thanh toán", "5. Thành công"]
user_counts = [119073, 69355, 28155, 19296, 12803]

fig_funnel = go.Figure(go.Funnel(
    y = stages_labels,
    x = user_counts,
    textinfo = "value+percent initial+percent previous",
    hoverinfo = "x+y+percent initial",
    marker = {"color": THEME_COLORS['funnel'], "line": {"width": [4, 2, 2, 2, 4], "color": "white"}},
    connector = {"line": {"color": "royalblue", "dash": "dot", "width": 1}}
))

fig_funnel.update_layout(
    title_text="<b>PHỄU CHUYỂN ĐỔI TỔNG THỂ</b><br><span style='font-size:12px'>Tỷ lệ giữ chân qua từng giai đoạn hành trình khách hàng</span>",
    plot_bgcolor=THEME_COLORS['background'],
    font=dict(family="Arial", size=14, color=THEME_COLORS['text']),
    margin=dict(l=100, r=100, t=100, b=50)
)
fig_funnel.show()

# =======================================================================
# 2. BIỂU ĐỒ SANKEY TINH TẾ (REFINED SANKEY)
# =======================================================================
sankey_data = [
    ('Paid Ads', 'Di động', 'Bỏ ngang', 27645),
    ('Organic', 'Di động', 'Bỏ ngang', 18047),
    ('Social', 'Di động', 'Bỏ ngang', 10707),
    ('Paid Ads', 'Máy tính', 'Bỏ ngang', 11928),
    ('Paid Ads', 'Di động', 'Thành công', 5398) # Ví dụ thêm luồng thành công
]

nodes = sorted(list(set([x[0] for x in sankey_data] + [x[1] for x in sankey_data] + [x[2] for x in sankey_data])))
node_map = {name: i for i, name in enumerate(nodes)}

# Định nghĩa màu cho từng nút (Nodes)
node_colors = []
for node in nodes:
    if node == 'Bỏ ngang': node_colors.append('#e74c3c')
    elif node == 'Thành công': node_colors.append('#2ecc71')
    elif node in ['Di động', 'Máy tính']: node_colors.append('#95a5a6')
    else: node_colors.append('#3498db')

fig_sankey = go.Figure(data=[go.Sankey(
    node = dict(pad=20, thickness=20, line=dict(color="white", width=2), label=nodes, color=node_colors),
    link = dict(
        source = [node_map[x[0]] for x in sankey_data for _ in range(2)],
        target = [node_map[x[1]] if i%2==0 else node_map[x[2]] for i, x in enumerate(sankey_data * 2)],
        value = [x[3] for x in sankey_data for _ in range(2)],
        color = 'rgba(149, 165, 166, 0.2)' # Màu dây nối mờ ảo, sang trọng
    )
)])

fig_sankey.update_layout(title_text="<b>DÒNG CHẢY TRUY CẬP VÀ ĐIỂM MA SÁT</b>", font_size=12, margin=dict(t=80))
fig_sankey.show()

# =======================================================================
# 3. BIỂU ĐỒ DOANH THU THẤT THOÁT (REVENUE LEAKAGE)
# =======================================================================
stages_leakage = ["Truy cập -> Xem", "Xem -> Giỏ", "Giỏ -> Thanh toán", "Thanh toán -> Mua"]
lost_revenue = [192576210, 159582844, 34314185, 25149791]

fig_leakage = px.bar(
    x=stages_leakage, y=lost_revenue,
    text_auto='.2s', # Rút gọn số (ví dụ 192M)
    title='<b>ƯỚC TÍNH DOANH THU THẤT THOÁT</b>',
    labels={'x': '', 'y': 'Số tiền (USD)'}
)

fig_leakage.update_traces(marker_color='#c0392b', opacity=0.8)
fig_leakage.add_hline(y=49590809, line_dash="dot", line_color="#27ae60", annotation_text="Doanh thu thực tế")
fig_leakage.update_layout(plot_bgcolor='white', xaxis={'categoryorder':'total descending'})
fig_leakage.show()

# =======================================================================
# 4. SO SÁNH PHÂN KHÚC NÂNG CAO (DUAL AXIS BAR-LINE)
# =======================================================================
segments = ["Mới", "Quay lại"]
conv_rates = [0.0542, 0.2062]
aov_values = [2598, 4494]

fig_seg = make_subplots(specs=[[{"secondary_y": True}]])
fig_seg.add_trace(go.Bar(x=segments, y=conv_rates, name="Tỷ lệ chuyển đổi (%)", marker_color='#3498db'), secondary_y=False)
fig_seg.add_trace(go.Scatter(x=segments, y=aov_values, name="Giá trị đơn (AOV)", mode='lines+markers+text', text=[f"${x}" for x in aov_values], textposition="top center", line=dict(color='#e67e22', width=4)), secondary_y=True)

fig_seg.update_layout(title_text="<b>HIỆU SUẤT THEO LOẠI KHÁCH HÀNG</b>", barmode='group', plot_bgcolor='white')
fig_seg.update_yaxes(title_text="Tỷ lệ chuyển đổi", tickformat=".1%", secondary_y=False, showgrid=False)
fig_seg.update_yaxes(title_text="Giá trị đơn hàng ($)", secondary_y=True, showgrid=False)
fig_seg.show()

BÁO CÁO TỐI ƯU HÓA CHUYỂN ĐỔI PHỄU D2C
Dự án: Hệ thống phân tích hành vi khách hàng

1. TỔNG QUAN HIỆU SUẤT PHỄU
Hệ thống ghi nhận 119.073 lượt truy cập, nhưng chỉ có 12.803 đơn hàng thành công.

Tỷ lệ chuyển đổi tổng thể (Overall CR): ~11%.

Điểm ma sát lớn nhất (Biggest Bottleneck): Giai đoạn từ Xem sản phẩm -> Thêm vào giỏ hàng. Chỉ có 41% người xem sản phẩm thực hiện bỏ hàng vào giỏ. Đây là nơi chúng ta mất đi lượng khách hàng tiềm năng lớn nhất.

2. ƯỚC TÍNH THIỆT HẠI KINH TẾ (REVENUE LEAKAGE)
Con số thất thoát thực sự gây "choáng váng" khi đặt cạnh doanh thu thực tế:

Doanh thu thực tế: ~50 triệu USD (đường kẻ xanh).

Doanh thu "bốc hơi" tại bước đầu (Truy cập -> Xem): 190 triệu USD.

Doanh thu "bốc hơi" tại bước giữa (Xem -> Giỏ hàng): 160 triệu USD.

Insight: Tổng số tiền chúng ta đánh mất ở phần trên của phễu (Upper Funnel) lớn gấp gần 7 lần doanh thu hiện tại. Chỉ cần cải thiện 5% tỷ lệ giữ chân ở các bước này, doanh thu có thể tăng trưởng vượt bậc.

3. PHÂN TÍCH DÒNG CHẢY VÀ PHÂN KHÚC
Nguồn rò rỉ: Sơ đồ Sankey chỉ rõ Paid Ads (Quảng cáo trả phí) đang mang lại lượng truy cập khổng lồ nhưng cũng là nguồn đóng góp nhiều nhất vào nhóm "Bỏ ngang", đặc biệt là trên thiết bị Di động.

Sức mạnh khách hàng cũ (Returning Users):

Khách hàng cũ có tỷ lệ chuyển đổi (20.6%) cao gấp gần 4 lần khách hàng mới (5.4%).

Giá trị đơn hàng (AOV) của khách cũ ($4494) cũng vượt trội so với khách mới ($2598).

CÁC INSIGHT RÚT RA (KEY INSIGHTS)
1. "Nỗi đau" mang tên Di động & Paid Ads
Lượng lớn ngân sách quảng cáo đang bị lãng phí trên thiết bị di động. Khách bấm vào quảng cáo nhưng thoát ra ngay (Bounce) hoặc không xem sản phẩm.

Nguyên nhân: Có thể do tốc độ tải trang trên di động chậm hoặc giao diện hiển thị sản phẩm chưa tối ưu cho màn hình nhỏ.

2. Trang Sản phẩm chưa đủ sức thuyết phục
Tỷ lệ rớt 59% từ bước Xem sản phẩm sang Bỏ giỏ là một tín hiệu báo động đỏ.

Nguyên nhân: Khách hàng đã quan tâm (đã xem), nhưng có thể giá cả chưa cạnh tranh, thiếu thông tin bảo hiểm/vận chuyển, hoặc nút "Thêm vào giỏ" khó tìm.

3. Khách hàng cũ là "mỏ vàng" lợi nhuận
Chúng ta đang chi tiền để tìm khách mới (với hiệu suất thấp) trong khi chưa khai thác hết tiềm năng từ khách cũ - những người sẵn sàng chi nhiều tiền hơn và dễ chốt đơn hơn.

ĐỀ XUẤT HÀNH ĐỘNG (ACTION PLAN)
Tối ưu Mobile UX: Kiểm tra lại ngay lập tức trải nghiệm người dùng trên điện thoại cho các chiến dịch Paid Ads. Ưu tiên cải thiện tốc độ load và quy trình xem xe.

Tăng cường "Social Proof" trên trang sản phẩm: Bổ sung đánh giá, hình ảnh thực tế và các cam kết uy tín để đẩy tỷ lệ Product View -> Add to Cart lên ít nhất 50%.

Chiến dịch Re-marketing: Tập trung ngân sách đeo bám những khách hàng đã từng thuê xe (Returning) bằng các ưu đãi đặc quyền, vì đây là tệp khách hàng mang lại ROI cao nhất.

Cứu vãn giỏ hàng (Cart Recovery): Với 34 triệu USD thất thoát ở bước thanh toán, cần triển khai ngay Email/Notification nhắc nhở khách hàng hoàn tất đơn hàng trong vòng 24h.